## הטמעות

בדוגמה הקודמת שלנו, פעלנו על וקטורי שקית מילים מממד גבוה עם אורך `vocab_size`, והמרנו במפורש מוקטורי ייצוג מיקום מממד נמוך לייצוג חד-חם מפוזר. ייצוג חד-חם זה אינו יעיל זיכרון, בנוסף, כל מילה מטופלת באופן עצמאי משל האחרות, כלומר וקטורי חד-חם המקודדים אינם מבטאים שום דמיון סמנטי בין מילים.

ביחידה זו, נמשיך לחקור את מערך הנתונים של **News AG**. כדי להתחיל, נטען את הנתונים ונתח כמה הגדרות מהמחברת הקודמת.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## מה זה הטמעה?

הרעיון של **הטמעה** הוא לייצג מילים על ידי וקטורים בצפיפות נמוכה מממדית, שמשקפים במידה כלשהי את המשמעות הסמנטית של מילה. נדון מאוחר יותר כיצד לבנות הטמעות מילים משמעותיות, אבל כרגע נחשוב על הטמעות כדרך להוריד את הממדיות של וקטור מילה.

ולכן, שכבת הטמעה תקבל מילה כקלט, ותפיק וקטור פלט בגודל `embedding_size` שהוגדר מראש. במובן מסוים, היא מאוד דומה לשכבת `Linear`, אך במקום לקבל וקטור מקודד בתהליך אחד-חם, היא תוכל לקבל מספר מילה כקלט.

על ידי שימוש בשכבת הטמעה כשכבה ראשונה ברשת שלנו, נוכל לעבור מדגם תיק מילים למודל **תיק הטמעות**, שבו תחילה נהפוך כל מילה בטקסט להטמעה המתאימה, ואז נחשב פונקציית אגירה על כל ההטמעות האלה, כמו `sum`, `average` או `max`.

![תמונה המראה מסווג הטמעות עבור חמש מילים ברצף.](../../../../../translated_images/he/embedding-classifier-example.b77f021a7ee67eee.webp)

רשתנו הנוירונית למיון תתחיל עם שכבת הטמעה, לאחר מכן שכבת אגירה, ומסווג לינארי מעליה:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### התמודדות עם גודל רצף משתנה

כתוצאה מהארכיטקטורה הזו, יש ליצור מיני-אצוות לרשת שלנו בצורה מסוימת. ביחידה הקודמת, כששימשנו בשקית מילים, כל הטנסורים של BoW במיני-אצווה היו באותו גודל `vocab_size`, ללא תלות באורך הממשי של רצף הטקסט שלנו. כשנעבור לייצוגי מילים, נקבל מספר משתנה של מילים בכל דוגמת טקסט, וכאשר נשלב את הדוגמאות הללו למיני-אצוות נצטרך להוסיף ריפוד.

ניתן לעשות זאת באמצעות אותה טכניקה של מתן פונקציית `collate_fn` למקור הנתונים:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### אימון מסווג מוטמע

עכשיו כשהגדרנו לוד מגובה נכון, נוכל לאמן את המודל באמצעות פונקציית האימון שהגדרנו ביחידה הקודמת:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **הערה**: אנחנו מאמנים כאן רק על 25 אלף רשומות (פחות מאפוך מלא אחד) למען הזמן, אבל אתם יכולים להמשיך באימון, לכתוב פונקציה לאימון למספר אפוקים, ולנסות את פרמטר קצב הלמידה כדי להגיע לדיוק גבוה יותר. אתם אמורים להיות מסוגלים להגיע לדיוק של כ-90%.


### שכבת EmbeddingBag וייצוג רצפים באורך משתנה

בארכיטקטורה הקודמת, נדרשנו למלא את כל הרצפים לאורך אחיד כדי להתאים אותם למיני-באטץ'. זו לא הדרך היעילה ביותר לייצוג רצפים באורך משתנה - גישה נוספת תהיה להשתמש בוקטור **offset**, אשר יחזיק את ההיסטים של כל הרצפים המאוחסנים בווקטור אחד גדול.

![דימוי המציג ייצוג רצף באמצעות ההיסט](../../../../../translated_images/he/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **הערה**: בתמונה למעלה, אנו מציגים רצף של תווים, אך בדוגמה שלנו אנו עובדים עם רצפים של מילים. עם זאת, העיקרון הכללי של ייצוג רצפים באמצעות וקטור ההיסט נשאר זהה.

כדי לעבוד עם ייצוג היסט, אנו משתמשים בשכבת [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). היא דומה ל`Embedding`, אך מקבלת וקטור תוכן ווקטור היסט כקלט, והיא כוללת גם שכבת ממוצע, שניתן לממש כ-`mean`, `sum` או `max`.

להלן רשת מותאמת שמשתמשת ב-`EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

כדי להכין את מערך הנתונים לאימון, עלינו לספק פונקציית המרה שתכין את וקטור ההסטה:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

שים לב, שבשונה מכל הדוגמאות הקודמות, הרשת שלנו מקבלת עכשיו שני פרמטרים: וקטור הנתונים ווקטור ההיסט, שהם בגדלים שונים. באופן דומה, הטוען שלנו מספק לנו גם 3 ערכים במקום 2: גם וקטורי הטקסט וגם היסט מסופקים בתור תכונות. לכן, עלינו להתאים מעט את פונקציית האימון שלנו כדי להתמודד עם זה:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## הטמעות סמנטיות: Word2Vec

בדוגמה הקודמת שלנו, שכבת ההטמעה של המודל למדה למפות מילים לייצוג וקטורי, אך ייצוג זה לא היה בעל משמעות סמנטית רבה. יהיה נחמד ללמוד ייצוג וקטורי כזה, שבו מילים דומות או מילים נרדפות יתאימו לוקטורים הקרובים זה לזה מבחינת מרחק וקטורי כלשהו (למשל מרחק אוקלידי).

כדי לעשות זאת, עלינו לאמן מראש את מודל ההטמעה שלנו על אוסף גדול של טקסט בצורה מסוימת. אחת הדרכים הראשונות לאימון הטמעות סמנטיות נקראת [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). היא מבוססת על שתי ארכיטקטורות עיקריות ששימושיות להפקת ייצוג מפוזר של מילים:

 - **תיק נשיאה רציף של מילים** (CBoW) — בארכיטקטורה זו, אנו מאמנים את המודל לנבא מילה מהקשר סובב. בהינתן הנגרם $(W_{-2},W_{-1},W_0,W_1,W_2)$, מטרת המודל היא לנבא את $W_0$ מתוך $(W_{-2},W_{-1},W_1,W_2)$.
 - **סיפ-גרם רציף** הוא הפוך ל-CBoW. המודל משתמש בחלון ההקשר הסובב את המילה כדי לנבא את המילה הנוכחית.

CBoW מהיר יותר, בעוד סיפ-גרם איטי יותר, אבל עושה עבודה טובה יותר בייצוג מילים נדירות.

![תמונה המציגה הן את האלגוריתמים CBoW והן את סיפ-גרם להמרת מילים לוקטורים.](../../../../../translated_images/he/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

כדי להתנסות בהטמעת word2vec שאומנה מראש על מאגר Google News, נוכל להשתמש בספריית **gensim**. להלן המילים הדומות ביותר ל'neural'

> **הערה:** כשהינך יוצר וקטורי מילים לראשונה, הורדתם עלולה לקחת זמן מה!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


אנו יכולים גם לחשב ייצוג וקטורי מהמילה, לשימוש באימון מודל סיווג (אנו מציגים רק את 20 המרכיבים הראשונים של הוקטור לשם בהירות):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

הדבר הנהדר ב-embeddings סמנטיים הוא שניתן לתמרן את קידוד הוקטור כדי לשנות את המשמעות. לדוגמה, אנחנו יכולים לבקש למצוא מילה, שהייצוג הווקטורי שלה יהיה קרוב ככל האפשר למילים *king* ו-*woman*, ורחוק מהמילה *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

הן CBoW והן Skip-Grams הן הטמעות "חיזוי", בכך שהן לוקחות בחשבון רק הקשרים מקומיים. Word2Vec אינו מנצל את ההקשר העולמי. 

**FastText**, מתבסס על Word2Vec על ידי לימוד ייצוגים וקטוריים עבור כל מילה ונ-גרמים של תווים שנמצאים בתוך כל מילה. ערכי הייצוגים לאחר מכן ממוצעים לקטור אחד בכל שלב אימון. אף שזה מוסיף הרבה חישובים נוספים לאימון מוקדם, זה מאפשר להטמעות מילים לקודד מידע תחת-מילה. 

שיטה נוספת, **GloVe**, מנצלת את רעיון מטריצת ההתארגנות המשותפת, משתמשת בשיטות נוירוניות לפרק את מטריצת ההתארגנות המשותפת לוקטורים של מילים שהם יותר ביטוייים ולא ליניאריים.

ניתן לשחק עם הדוגמה על ידי שינוי ההטמעות ל-FastText ו-GloVe, כי gensim תומך בכמה מודלים שונים של הטמעות מילים.


## שימוש בהתממשק מראש ב-PyTorch

ניתן לשנות את הדוגמה שלמעלה כדי למלא מראש את המטריצה בשכבת ההתממשקות שלנו בהתממשקים סמנטיים, כגון Word2Vec. עלינו לקחת בחשבון שיתכן שהאוצרות של ההתממשקות האמורה וטקסט הקורפוס שלנו לא יתאימו, לכן נאתחל משקלים עבור המילים החסרות בערכים אקראיים:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


עכשיו נחנך את המודל שלנו. שימו לב שהזמן שלוקח לאמן את המודל הוא משמעותית גדול יותר מאשר בדוגמה הקודמת, בגלל גודל שכבת ההטמעה הגדולה יותר, ולכן מספר הפרמטרים הגבוה בהרבה. גם, בגלל זה, ייתכן ונצטרך לאמן את המודל שלנו על יותר דוגמאות אם אנחנו רוצים למנוע התאמה יתרה.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

במקרה שלנו אנחנו לא רואים עלייה גדולה בדיוק, ככל הנראה בגלל אוצר מילים שונה במידה רבה. 
כדי להתגבר על הבעיה של אוצר מילים שונה, נוכל להשתמש באחת מהפתרונות הבאים:
* לאמן מחדש את מודל word2vec על אוצר המילים שלנו
* לטעון את מערכת הנתונים שלנו עם אוצר מילים מהמודל word2vec שאומן מראש. אוצר המילים המשמש לטעינת מערכת הנתונים ניתן לציון בזמן הטעינה.

הגישה השנייה נראית קלה יותר, בעיקר כי מסגרת PyTorch `torchtext` כוללת תמיכה מובנית באמבדים. לדוגמה, נוכל לאתחל אוצר מילים מבוסס GloVe באופן הבא:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


אוצר המילים שטען כולל את הפעולות הבסיסיות הבאות:
* `vocab.stoi` מילון מאפשר לנו להמיר מילה למספר האינדקס שלה במילון
* `vocab.itos` עושה את ההפך - ממיר מספר למילה
* `vocab.vectors` הוא מערך וקטורי הטמעה, כך שכדי לקבל את הטמעת המילה `s` אנו צריכים להשתמש ב-`vocab.vectors[vocab.stoi[s]]`

הנה דוגמה של טיפול בהטמעות להמחיש את המשוואה **kind-man+woman = queen** (הייתי צריך לכוונן מעט את המקדם כדי שזה יעבוד):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

כדי לאמן את הממיין באמצעות ההטמעות הללו, אנחנו צריכים קודם לקודד את מערך הנתונים שלנו באמצעות אוצר המילים של GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

כפי שראינו למעלה, כל האמבדינגים הוקטוריים נשמרים במטריצת `vocab.vectors`. זה עושה את זה קל מאוד לטעון את המשקלים הללו למשקלים של שכבת האמבדינג בעזרת העתקה פשוטה:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

עכשיו נאמן את המודל שלנו ונראה אם נקבל תוצאות טובות יותר:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

אחת הסיבות לכך שאיננו רואים עלייה משמעותית בדיוק היא בגלל שמילים מסוימות ממאגר הנתונים שלנו חסרות באוצר מילים המאומנים מראש של GloVe, ולכן הן למעשה מתעלמות. כדי להתגבר על עובדה זו, אנו יכולים לאמן את ההטמעות שלנו על מאגר הנתונים שלנו. 


## הטמעות הקשריות

מגבלה מרכזית אחת של ייצוגי הטמעות מראש מסורתיים כגון Word2Vec היא בעיית היבחנות משמעויות מילים. בעוד שהטמעות מראש יכולות ללכוד חלק מהמשמעות של מילים בהקשר, כל משמעות אפשרית של מילה מקודדת באותה הטמעה. זה יכול לגרום לבעיות במודלים המשך, שכן מילים רבות כמו המילה 'play' יש להן משמעויות שונות בהתאם להקשר בו הן משמשות.

לדוגמה, למילה 'play' בשתי משפטים שונים אלה יש משמעות שונה למדי:
- הלכתי להצגה בתיאטרון.
- ג'ון רוצה לשחק עם חבריו.

ההטמעות המוכנות לעיל מייצגות את שתי המשמעויות האלו של המילה 'play' באותה הטמעה. כדי להתגבר על מגבלה זו, עלינו לבנות הטמעות מבוססות על **מודל שפה**, אשר מאומן על מסה גדולה של טקסט, ו*יודע* כיצד מילים יכולות להיות משולבות בהקשרים שונים. דיון בהטמעות הקשריות אינו בתחום מדריך זה, אך נחזור אליהן כשנדבר על מודלי שפה ביחידה הבאה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
